# Notebook 5: HTTP Feature Engineering

## Objective

In this notebook, we will extract behavioral features from the HTTP log of the CERT R6.2 dataset.

The HTTP dataset records every web activity performed by employees, such as:

- WWW Visit
- WWW Download
- WWW Upload

These activities help identify abnormal browsing behavior that may indicate insider threats.

---

## Features to Extract

For every employee, we will compute:

- Total HTTP events
- Total website visits
- Total downloads
- Total uploads
- After-hours HTTP activity
- Weekend HTTP activity
- Number of unique URLs visited
- Number of unique PCs used for browsing

---

## Why Chunk Processing?

The HTTP dataset contains millions of records.

Loading the entire dataset into memory causes:

- ArrowMemoryError
- MemoryError

Therefore, we process the dataset in chunks of 100,000 rows.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
from collections import defaultdict

# Define Dataset Paths

Here we define:

- DATA_PATH → Location of the CERT dataset
- FEATURE_PATH → Folder where extracted features will be saved

If the feature folder does not exist, it will be created automatically.

In [2]:
DATA_PATH = Path("../DATA/raw/CERT_R6.2/data")
FEATURE_PATH = Path("../DATA/features")

FEATURE_PATH.mkdir(parents=True, exist_ok=True)

# Verify Dataset

Before processing the dataset, we verify:

- Current working directory
- Whether the HTTP dataset exists

This helps avoid file path errors.

In [3]:
print("Current Working Directory:")
print(Path.cwd())

print("\nChecking HTTP Dataset...")

if (DATA_PATH / "http.csv").exists():
    print("✅ http.csv Found")
else:
    print("❌ http.csv Not Found")

Current Working Directory:
d:\OPENCODE\insider-threat-detection\NOTEBOOKS

Checking HTTP Dataset...
✅ http.csv Found


# Create Chunk Reader

Instead of loading the complete dataset into memory,
we read only 100,000 rows at a time.

This technique is called **Chunk Processing**.

Benefits:

- Very low RAM usage
- Faster processing
- Prevents ArrowMemoryError

In [4]:
chunks = pd.read_csv(
    DATA_PATH / "http.csv",
    chunksize=100000
)

print("✅ Chunk Reader Created Successfully")

✅ Chunk Reader Created Successfully


# Initialize Data Structures

We create:

- http_profiles → stores chunk-wise feature tables
- user_urls → stores unique URLs for each employee
- user_pcs → stores unique PCs used by each employee

defaultdict(set) automatically keeps only unique values.

In [5]:
http_profiles = []

user_urls = defaultdict(set)

user_pcs = defaultdict(set)

# Process Each HTTP Chunk

The HTTP dataset is too large to fit into memory.

Therefore, we process **100,000 rows at a time**.

For every chunk, we perform the following steps:

1. Convert the date column into datetime format.
2. Extract hour and weekday.
3. Detect after-hours activity.
4. Detect weekend activity.
5. Identify Visits, Downloads, and Uploads.
6. Create chunk-level user features.
7. Store unique URLs.
8. Store unique PCs.
9. Save the chunk summary.

This loop continues until the entire dataset has been processed.

In [6]:
# Process Every Chunk

for chunk_number, chunk in enumerate(chunks, start=1):

    print(f"Processing Chunk {chunk_number}")

    # --------------------------------------------------
    # Convert Date Column
    # --------------------------------------------------

    chunk["date"] = pd.to_datetime(
        chunk["date"],
        format="%m/%d/%Y %H:%M:%S"
    )

    # --------------------------------------------------
    # Extract Time Information
    # --------------------------------------------------

    chunk["hour"] = chunk["date"].dt.hour
    chunk["weekday"] = chunk["date"].dt.weekday

    # --------------------------------------------------
    # After Hours Activity
    # Working Hours : 08:00 AM - 05:59 PM
    # --------------------------------------------------

    chunk["after_hours"] = (
        (chunk["hour"] < 8) |
        (chunk["hour"] >= 18)
    ).astype(int)

    # --------------------------------------------------
    # Weekend Activity
    # Saturday = 5
    # Sunday = 6
    # --------------------------------------------------

    chunk["weekend_activity"] = (
        chunk["weekday"] >= 5
    ).astype(int)

    # --------------------------------------------------
    # HTTP Activity Types
    # --------------------------------------------------

    chunk["is_visit"] = (
        chunk["activity"] == "WWW Visit"
    ).astype(int)

    chunk["is_download"] = (
        chunk["activity"] == "WWW Download"
    ).astype(int)

    chunk["is_upload"] = (
        chunk["activity"] == "WWW Upload"
    ).astype(int)

    # --------------------------------------------------
    # Create User Features for Current Chunk
    # --------------------------------------------------

    chunk_profile = (
        chunk
        .groupby("user")
        .agg(

            total_http_events=(
                "activity",
                "count"
            ),

            http_visit_count=(
                "is_visit",
                "sum"
            ),

            http_download_count=(
                "is_download",
                "sum"
            ),

            http_upload_count=(
                "is_upload",
                "sum"
            ),

            after_hours_http_count=(
                "after_hours",
                "sum"
            ),

            weekend_http_count=(
                "weekend_activity",
                "sum"
            )

        )
        .reset_index()
    )

    # --------------------------------------------------
    # Save Chunk Features
    # --------------------------------------------------

    http_profiles.append(chunk_profile)

    # --------------------------------------------------
    # Store Unique URLs
    # --------------------------------------------------

    for user, urls in chunk.groupby("user")["url"]:

        user_urls[user].update(
            urls.dropna().unique()
        )

    # --------------------------------------------------
    # Store Unique PCs
    # --------------------------------------------------

    for user, pcs in chunk.groupby("user")["pc"]:

        user_pcs[user].update(
            pcs.dropna().unique()
        )

print("\n✅ All Chunks Processed Successfully")

Processing Chunk 1
Processing Chunk 2
Processing Chunk 3
Processing Chunk 4
Processing Chunk 5
Processing Chunk 6
Processing Chunk 7
Processing Chunk 8
Processing Chunk 9
Processing Chunk 10
Processing Chunk 11
Processing Chunk 12
Processing Chunk 13
Processing Chunk 14
Processing Chunk 15
Processing Chunk 16
Processing Chunk 17
Processing Chunk 18
Processing Chunk 19
Processing Chunk 20
Processing Chunk 21
Processing Chunk 22
Processing Chunk 23
Processing Chunk 24
Processing Chunk 25
Processing Chunk 26
Processing Chunk 27
Processing Chunk 28
Processing Chunk 29
Processing Chunk 30
Processing Chunk 31
Processing Chunk 32
Processing Chunk 33
Processing Chunk 34
Processing Chunk 35
Processing Chunk 36
Processing Chunk 37
Processing Chunk 38
Processing Chunk 39
Processing Chunk 40
Processing Chunk 41
Processing Chunk 42
Processing Chunk 43
Processing Chunk 44
Processing Chunk 45
Processing Chunk 46
Processing Chunk 47
Processing Chunk 48
Processing Chunk 49
Processing Chunk 50
Processin

# Merge Chunk Features

Each chunk contains user-level HTTP features.

Since the same user can appear in multiple chunks, we need to:

1. Combine all chunk summaries.
2. Merge duplicate users.
3. Sum all feature values.

This produces one final row for each user.

In [7]:
# Combine All Chunk Profiles

http_profile = pd.concat(
    http_profiles,
    ignore_index=True
)

print("Rows Before User Merge:", len(http_profile))

http_profile.head()

Rows Before User Merge: 4420795


,user,total_http_events,http_visit_count,http_download_count,http_upload_count,after_hours_http_count,weekend_http_count
0,AAB0162,30,30,0,0,3,0
1,AAB0398,21,21,0,0,5,0
2,AAC0610,7,7,0,0,1,0
3,AAC0668,15,15,0,0,1,0
4,AAC3270,2,2,0,0,0,0


# Aggregate Duplicate Users

The same employee may appear in multiple chunks.

We group by the user column and sum all numerical features to obtain one final record per employee.

In [8]:
http_profile = (
    http_profile
    .groupby("user", as_index=False)
    .sum(numeric_only=True)
)

print("Unique Users:", len(http_profile))

http_profile.head()

Unique Users: 4000


,user,total_http_events,http_visit_count,http_download_count,http_upload_count,after_hours_http_count,weekend_http_count
0,AAB0162,33725,33725,0,0,4277,0
1,AAB0398,40584,40584,0,0,7128,0
2,AAC0610,10324,10324,0,0,240,0
3,AAC0668,33820,33820,0,0,320,0
4,AAC3270,3560,3560,0,0,0,0


# Calculate Unique URLs

During chunk processing, we stored every unique URL visited by each employee.

Now we calculate the total number of unique URLs for every user.

In [9]:
unique_url_df = pd.DataFrame({

    "user": list(user_urls.keys()),

    "unique_urls": [
        len(urls)
        for urls in user_urls.values()
    ]

})

print("Unique URL Features")

unique_url_df.head()

Unique URL Features


,user,unique_urls
0,AAB0162,144
1,AAB0398,351
2,AAC0610,267
3,AAC0668,227
4,AAC3270,60


# Calculate Unique PCs

Similarly, we calculate how many different computers each employee used while accessing websites.

In [10]:
unique_pc_df = pd.DataFrame({

    "user": list(user_pcs.keys()),

    "unique_http_pcs": [
        len(pcs)
        for pcs in user_pcs.values()
    ]

})

print("Unique PC Features")

unique_pc_df.head()

Unique PC Features


,user,unique_http_pcs
0,AAB0162,1
1,AAB0398,1
2,AAC0610,1
3,AAC0668,1
4,AAC3270,1


# Merge All HTTP Features

Now we combine:

- Chunk-based HTTP statistics
- Unique URL counts
- Unique PC counts

This creates the final HTTP feature table.

In [11]:
# Merge URL Features

http_profile = http_profile.merge(

    unique_url_df,

    on="user",

    how="left"

)

# Merge PC Features

http_profile = http_profile.merge(

    unique_pc_df,

    on="user",

    how="left"

)

# Replace Missing Values

http_profile = http_profile.fillna(0)

print("Missing Values")

print(http_profile.isnull().sum())

Missing Values
user                      0
total_http_events         0
http_visit_count          0
http_download_count       0
http_upload_count         0
after_hours_http_count    0
weekend_http_count        0
unique_urls               0
unique_http_pcs           0
dtype: int64


# Verify Final Dataset

Let's inspect the final dataset before saving it.

We verify:

- Number of rows
- Number of columns
- Data types
- Sample records

In [12]:
print("Final Dataset Shape:")

print(http_profile.shape)

print("\nDataset Information")

http_profile.info()

print("\nFirst Five Rows")

http_profile.head()

Final Dataset Shape:
(4000, 9)

Dataset Information
<class 'pandas.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   user                    4000 non-null   str  
 1   total_http_events       4000 non-null   int64
 2   http_visit_count        4000 non-null   int64
 3   http_download_count     4000 non-null   int64
 4   http_upload_count       4000 non-null   int64
 5   after_hours_http_count  4000 non-null   int64
 6   weekend_http_count      4000 non-null   int64
 7   unique_urls             4000 non-null   int64
 8   unique_http_pcs         4000 non-null   int64
dtypes: int64(8), str(1)
memory usage: 309.2 KB

First Five Rows


,user,total_http_events,http_visit_count,http_download_count,http_upload_count,after_hours_http_count,weekend_http_count,unique_urls,unique_http_pcs
0,AAB0162,33725,33725,0,0,4277,0,144,1
1,AAB0398,40584,40584,0,0,7128,0,351,1
2,AAC0610,10324,10324,0,0,240,0,267,1
3,AAC0668,33820,33820,0,0,320,0,227,1
4,AAC3270,3560,3560,0,0,0,0,60,1


# Save HTTP Features

The final HTTP feature dataset is saved inside the `features` folder.

This dataset will be used later when combining features from:

- Logon
- Device
- Email
- File
- Psychometric
- LDAP

to build the complete Insider Threat Detection dataset.

In [13]:
output_file = FEATURE_PATH / "http_features.csv"

http_profile.to_csv(

    output_file,

    index=False

)

print(f"✅ HTTP Features Saved Successfully!")

print(f"Location: {output_file}")

✅ HTTP Features Saved Successfully!
Location: ..\DATA\features\http_features.csv


# Verify Saved File

Finally, we reload the saved file to ensure it has been written correctly.

This is a good practice before moving to the next notebook.

In [14]:
saved_http = pd.read_csv(output_file)

print("Saved Dataset Shape:")

print(saved_http.shape)

saved_http.head()

Saved Dataset Shape:
(4000, 9)


,user,total_http_events,http_visit_count,http_download_count,http_upload_count,after_hours_http_count,weekend_http_count,unique_urls,unique_http_pcs
0,AAB0162,33725,33725,0,0,4277,0,144,1
1,AAB0398,40584,40584,0,0,7128,0,351,1
2,AAC0610,10324,10324,0,0,240,0,267,1
3,AAC0668,33820,33820,0,0,320,0,227,1
4,AAC3270,3560,3560,0,0,0,0,60,1
